# 🧬 Cellular Reasoning Fabric (CRF) - GPU Training & Benchmarking Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yasirusman85/pc-ai/blob/master/notebooks/crf_gpu_colab.ipynb)

This notebook runs the **Cellular Reasoning Fabric (CRF)** model and benchmarks it against a parameter-matched **Transformer** on GPU (T4 / V100 / A100).

## Step 1: Environment Setup & Clone Repository

In [ ]:
# Clone repo if running in Google Colab
import os
if not os.path.exists('src/crf_reasoning'):
    !git clone https://github.com/yasirusman85/pc-ai.git
    %cd pc-ai

# Install dependencies
!pip install -r requirements.txt
!pip install -e .

## Step 2: Verify GPU Acceleration

In [ ]:
import torch
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))
    print("Device Memory (GB):", f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.2f}")
else:
    print("⚠️ Running on CPU! Enable GPU via Runtime -> Change runtime type -> T4 GPU in Colab.")

## Step 3: Run Full Benchmark Suite (CRF vs Transformer)

In [ ]:
# Set device based on availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running benchmark on: {device}")

# Run the official benchmark workflow
!python scripts/benchmark.py

## Step 4: Visualize Perplexity & Dynamics Results

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

with open('results/results.json') as f:
    res = json.load(f)

data = []
comparison = res.get('main_comparison', {})
for benchmark, val in comparison.items():
    crf_ppl = val['crf']['best_val_ppl']
    tf_ppl = val['transformer']['best_val_ppl']
    advantage = tf_ppl / crf_ppl if crf_ppl > 0 else 0
    data.append({
        'Benchmark': benchmark.capitalize(),
        'CRF PPL': round(crf_ppl, 2),
        'Transformer PPL': round(tf_ppl, 2),
        'CRF Advantage': f"{advantage:.2f}x"
    })

df = pd.DataFrame(data)
print("=== HEAD-TO-HEAD BENCHMARK RESULTS ===")
display(df)

# Bar chart plot
plt.figure(figsize=(10, 5))
x = range(len(df))
plt.bar([i - 0.2 for i in x], df['CRF PPL'], width=0.4, label='CRF', color='#8b5cf6')
plt.bar([i + 0.2 for i in x], df['Transformer PPL'], width=0.4, label='Transformer', color='#3b82f6')
plt.xticks(x, df['Benchmark'])
plt.ylabel('Validation Perplexity (Lower is Better)')
plt.title('CRF vs Transformer Perplexity')
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

## Step 5: Test Live Cell Dynamics (Splits, Deaths, Merges)

In [ ]:
import sys
sys.path.insert(0, 'src')

import torch
from crf_reasoning.crf_vectorized import CRFLanguageModel, AblationConfig

# Initialize CRF with CUDA
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg = AblationConfig()
model = CRFLanguageModel(
    vocab_size=1000,
    d_model=128,
    d_hidden=64,
    n_init_cells=32,
    max_cells=256,
    n_crf_steps=8,
    k_neighbors=4,
    cfg=cfg,
).to(dev)

# Input batch
x = torch.randint(0, 1000, (4, 32), device=dev)
logits, loss, metrics = model(x, targets=x, collect_metrics=True)

print(f"Device: {dev}")
print(f"Model Parameters: {model.n_params:,}")
print(f"Loss: {loss.item():.4f}")
print(f"Splits: {metrics.n_splits} | Merges: {metrics.n_merges} | Deaths: {metrics.n_deaths}")
print(f"Energy Mean: {metrics.energy_mean:.3f} | Max: {metrics.energy_max:.3f}")